# Hyperparameter Tuning for PlainNN

In [1]:
import time
import numpy as np
from pathlib import Path
import pandas as pd
import sys
import wandb
import pickle

import gc
import torch

sys.path.insert(0, "../../")
from geometric_extraction_helper import ALL_FEATURE_KEYS
from models_helper import ModelTrainer
from nn_models import PlainNN, NNClassifier

from env_helper import get_env_var
from wandb_helper import get_sweep_params, set_run_name, start_training

WANDB_PROJECT = get_env_var("WANDB_PROJECT")
WANDB_ENTITY  = get_env_var("WANDB_ENTITY")

/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [2]:
# load data
DATA_DIR = "../../3_data_curation_enrichement/3_9_split dataset to datasets and remove rare classes"

df_train = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed.parquet")
df_train_over = pd.read_parquet(Path(DATA_DIR) / "dataset_train_rare_classes_removed_oversampled.parquet")
df_val = pd.read_parquet(Path(DATA_DIR) / "dataset_validation_rare_classes_removed.parquet")

df_train.head()

,aabb_min_x,aabb_min_y,aabb_min_z,aabb_max_x,aabb_max_y,aabb_max_z,aabb_len_x,aabb_len_y,aabb_len_z,aabb_ratio_z_x,...,label_ifc_entity,label_predefined_type,label_is_external,label_load_bearing,minor_label_unter_terrain,minor_label_konstruktionsergaenzung,minor_label_deckbelag,minor_label_bekleidung,minor_label_aussenliegendes_bauteil,minor_label_sonnenschutz
0,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
1,0.000000,0.00,0.0,2.260000,2.310003,0.85,2.260000,2.310003,0.85,0.376106,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
2,-0.752578,0.00,0.0,9.893693,14.360000,2.82,10.646271,14.360000,2.82,0.264881,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
3,0.000000,-1.85,0.0,10.646270,12.510000,2.82,10.646270,14.360000,2.82,0.264882,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown
4,-0.158796,0.00,0.0,9.394583,9.845000,0.24,9.553379,9.845000,0.24,0.025122,...,IfcSpace,GFA,unknown,unknown,unknown,unknown,unknown,unknown,unknown,unknown


In [3]:
# get labels
label_cols = [c for c in df_train.columns if c.startswith("label_")]

# get data splits
X_train      = df_train[ALL_FEATURE_KEYS]
y_train      = df_train[label_cols]
X_train_over = df_train_over[ALL_FEATURE_KEYS]
y_train_over = df_train_over[label_cols]
X_val        = df_val[ALL_FEATURE_KEYS]
y_val        = df_val[label_cols]

print(f"Train: {len(X_train)}, Train (oversampled): {len(X_train_over)}, Val: {len(X_val)}")
print(f"Labels: {label_cols}")

Train: 26977, Train (oversampled): 28756, Val: 8458
Labels: ['label_ifc_entity', 'label_predefined_type', 'label_is_external', 'label_load_bearing']


In [4]:
# number of classes per label
n_classes = [y_train_over[col].nunique() for col in label_cols]
n_classes

[13, 17, 3, 3]

## 1. Bayesian Hyperparameter Tuning for plain Neural Network

In [ ]:
# name of best model file
BEST_MODEL_PATH = "best_nn_model"

sweep_config_v1 = {
    "method": "bayes",
    "metric": {"name": "val/f1_macro", "goal": "maximize"},
    "parameters": {
        "n_hidden":            {"values": [128, 256, 512, 768, 1024]},
        "dropout":             {"min": 0.1,  "max": 0.5,  "distribution": "uniform" },
        "learning_rate":       {"min": 1e-8, "max": 1e-1, "distribution": "log_uniform_values"},
        "batch_size":          {"values": [64, 128, 256]},
        "epochs":              {"values": [200]},
        "training_oversample": {"values": [True, False]},
    }
}

best = {"score": -1}

def train_fn():
    with wandb.init() as run:
        wandb.log({"model_type": "plain_neural_network"})
        set_run_name(run, config_params=["n_hidden", "dropout", "learning_rate", "batch_size", "epochs", "training_oversample"])
        params = get_sweep_params(run)

        # get params and pop oversampling param
        oversample = params.pop("training_oversample", True)
        X_tr = X_train_over if oversample else X_train
        y_tr = y_train_over if oversample else y_train

        # log count of features and all features used for training
        wandb.log({"num_features": len(X_train_over.columns if oversample else X_train.columns)})
        wandb.log({"feature_names": list(X_train_over.columns if oversample else X_train.columns)})

        plain_nn = PlainNN(
            n_inputs=X_tr.shape[1],
            n_hidden=params.pop("n_hidden"),
            label_num_classes=n_classes,
            dropout=params.pop("dropout"),
        )
        model = NNClassifier(plain_nn, label_cols)
        trainer = ModelTrainer(model)

        def on_epoch_end(epoch, loss, train_metrics, val_metrics):
            log_data = {"train/loss": loss}
            for label, metrics in train_metrics.items():
                for metric, value in metrics.items():
                    log_data[f"train/{label}/{metric}"] = value
            log_data["train/f1_macro"] = round(np.mean([m["f1_macro"] for m in train_metrics.values()]), 4)
            for label, metrics in val_metrics.items():
                for metric, value in metrics.items():
                    log_data[f"val/{label}/{metric}"] = value
            log_data["val/f1_macro"] = round(np.mean([m["f1_macro"] for m in val_metrics.values()]), 4)
            wandb.log(log_data, step=epoch)

        # start training and measure duration
        t0 = time.time()
        trainer.fit(X_tr, y_tr, X_val=X_val, y_val=y_val, on_epoch_end=on_epoch_end, **params)
        train_duration_s = round(time.time() - t0, 2)

        wandb.log({"train_duration_s": train_duration_s})

        val_f1 = trainer.evaluate(X_val, y_val).loc["mean", "f1_macro"]
        if val_f1 > best["score"]:
            best["score"] = val_f1
            Path("./models").mkdir(exist_ok=True)
            current_name = f"./models/{BEST_MODEL_PATH}_{val_f1:.4f}.pkl"
            with open(current_name, "wb") as f:
                pickle.dump(trainer, f)
            print(f"New best model (val/f1_macro={val_f1:.4f}) saved to {current_name}")

        # cleanup
        del model
        del trainer

        # free up gpu memory
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()

start_training(
    train_fn,
    use_wandb=True,
    sweep_config=sweep_config_v1,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/lukasstoeckli/.netrc
wandb: Currently logged in as: luki-st (baa_ls) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: s8w8npdf
Sweep URL: https://wandb.ai/baa_ls/eBKPh_classifier/sweeps/s8w8npdf


wandb: Agent Starting Run: wxar60vg with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.47567901570475624
wandb: 	epochs: 500
wandb: 	lr: 8.658983990236811e-06
wandb: 	n_hidden: 1024
wandb: 	training_oversample: False
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/lukasstoeckli/.netrc.


Epoch   1/500, loss=127.1696, val f1_macro: label_ifc_entity=0.1101  label_predefined_type=0.0665  label_is_external=0.3469  label_load_bearing=0.4003, mean=0.2309
Epoch   2/500, loss=59.4670, val f1_macro: label_ifc_entity=0.1379  label_predefined_type=0.0923  label_is_external=0.5320  label_load_bearing=0.4089, mean=0.2928
Epoch   3/500, loss=40.8021, val f1_macro: label_ifc_entity=0.1464  label_predefined_type=0.1632  label_is_external=0.5868  label_load_bearing=0.4836, mean=0.3450
Epoch   4/500, loss=53.7369, val f1_macro: label_ifc_entity=0.1673  label_predefined_type=0.1178  label_is_external=0.6208  label_load_bearing=0.5102, mean=0.3540
Epoch   5/500, loss=58.5903, val f1_macro: label_ifc_entity=0.1736  label_predefined_type=0.1252  label_is_external=0.6369  label_load_bearing=0.4759, mean=0.3529
Epoch   6/500, loss=67.6691, val f1_macro: label_ifc_entity=0.1770  label_predefined_type=0.1342  label_is_external=0.6638  label_load_bearing=0.4727, mean=0.3619
Epoch   7/500, loss=4

wandb: Ctrl + C detected. Stopping sweep.
Traceback (most recent call last):
  File "/var/folders/f3/z549264j6mn3xvpzd9tj43cr0000gn/T/ipykernel_78533/4114872291.py", line 50, in train_fn
    trainer.fit(X_tr, y_tr, X_val=X_val, y_val=y_val, on_epoch_end=on_epoch_end, **params)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/../../models_helper.py", line 30, in fit
    self.pipeline.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, lr=lr, device=device, X_val=X_val, y_val=y_val, on_epoch_end=on_epoch_end)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/../../nn_models.py", line 120, in fit
    train_metrics = se

ConnectionResetError: Connection lost

Traceback (most recent call last):
  File "/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/wandb/agents/pyagent.py", line 306, in _run_job
    raise _JobError(f"Run threw exception: {str(e)}") from e
wandb.agents.pyagent._JobError: Run threw exception: [Errno 32] Broken pipe

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.13/3.13.12/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 1044, in _bootstrap_inner
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Cellar/python@3.13/3.13.12/Frameworks/Python.framework/Versions/3.13/lib/python3.13/threading.py", line 995, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/.venv/lib/python3.13/site-packages/wandb/agents/pyagent.py", line 311, in _run_job
    w

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x127acfb60>> (for post_run_cell), with arguments args (<ExecutionResult object at 1237b5a50, execution_count=5 error_before_exec=None error_in_exec=Connection lost info=<ExecutionInfo object at 10c31bc50, raw_cell="# name of best model file
BEST_MODEL_PATH = "best_.." transformed_cell="# name of best model file
BEST_MODEL_PATH = "best_.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/lukasstoeckli/GitLabProjects/hslu_baa_ebkph_classifier/code/4_modeling/4_5_hyperparameter_tuning/hyperparameter_tuning_nn.ipynb#X45sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe